<a href="https://colab.research.google.com/github/liviatcunha-fio/Comunica-oemSa-de/blob/main/Aula3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Instalar (se necessário)
install.packages("googledrive")

# Carregar biblioteca
library(googledrive)

# Autenticar
drive_auth()

# Criar pasta (exemplo)
library(googledrive)

drive_auth()

# verificar se já existe
if (nrow(drive_find("meu_projeto_r_drive")) == 0) {
  drive_create("meu_projeto_r_drive")
  print("Pasta criada.")
} else {
  print("A pasta já existe.")
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

The googledrive package is requesting access to your Google account.
Enter '1' to start a new auth process or select a pre-authorized account.
1: Send me to the browser for a new auth process.
2: liviat.cunha@gmail.com


Selection: 2


The googledrive package is requesting access to your Google account.
Enter '1' to start a new auth process or select a pre-authorized account.
1: Send me to the browser for a new auth process.
2: liviat.cunha@gmail.com


Selection: 2
[1] "A pasta já existe."


#Explicação do script acima
O script realizou a autenticação com o Google Drive pelo pacote googledrive. Depois verificou a existência da pasta utilizando a função drive_find(). Como a pasta já existia não foi necessário criá-la novamente.

#Prompt para identificar erros

Analise o script em R abaixo e identifique possíveis erros de sintaxe e lógica.
Para cada erro encontrado:
1. descreva o problema
2. explique por que ele ocorre
3. indique a linha aproximada
4. apresente a correção
Ao final, forneça a versão corrigida do script.

In [5]:
# Definir os pacotes necessários

required_packages <- c("dplyr", "ggplot2", "readr")

# verificar/instalar
for (pkg in required_packages) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
    library(pkg, character.only = TRUE)
  } else {
    message(paste("O pacote", pkg, "já está carregado."))
  }
}

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


O pacote dplyr já está carregado.

Loading required package: ggplot2

O pacote ggplot2 já está carregado.

Loading required package: readr

O pacote readr já está carregado.



# Explicação script acima

O bloco else foi adicionado para a condição em que o pacote já está instalado e carregado, evitando reinstalações e devolvendo uma mensagem informativa.

## Refatorar script criado em sala

### Porque ele funcinou junto?
Ele só funciona inteiro porque:
1. tudo está acoplado na main()
2. Há dependência de ordem rígida
3. Variáveis globais são criadas no final (assign)
Autenticação + configuração + uso estão misturados

Isso impede execução modular.

Abaixo script separado.

In [8]:
# INSTALAR
install.packages("googledrive")
install.packages("tidyverse")

# CARREGAR
library(googledrive)
library(tidyverse)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [9]:
#AUTENTIFICAR
autenticar_drive <- function() {
  drive_auth(email = TRUE, cache = ".secrets", use_oob = TRUE)

  if (!drive_has_token()) {
    stop("Falha na autenticação")
  }

  message("Autenticado com sucesso")
}

autenticar_drive()

Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=c7e14ef9b6c8ca0e0517dd07188d1551&access_type=offline&prompt=consent



Enter authorization code: eyJjb2RlIjoiNC8wQWNpOThFLVBSb0Vad3FoSXdtVDBXSE5jdWhtQmJCUktOYy04VUZvMWF2dE1UNE5YQUxCSjlzVkJjNnhrdk1Rd0dpRXI1QSIsInN0YXRlIjoiYzdlMTRlZjliNmM4Y2EwZTA1MTdkZDA3MTg4ZDE1NTEifQ==


Autenticado com sucesso



In [11]:
#CONFIGURAR
configurar_pasta <- function(Aula_3_de_R) {

  pasta <- drive_find(Aula_3_de_R)

  if (nrow(pasta) == 0) {
    message("Criando pasta...")
    pasta <- drive_mkdir(Aula_3_de_R)
  } else {
    message("Pasta já existe")
    pasta <- pasta[1,]
  }

  return(pasta)
}

project_folder <- configurar_pasta("Aula_3_de_R")

Criando pasta...

Created Drive file:

• Aula_3_de_R <id: 1bF_6molnX_6-NhhBhJagLQccG0sYEpkd>

With MIME type:

• application/vnd.google-apps.folder



In [12]:
#CRUD
#C

criar_arquivo <- function(pasta, nome, conteudo) {
  temp <- tempfile()

  writeLines(conteudo, temp)

  drive_upload(temp, path = pasta, name = nome)

  unlink(temp)
}

criar_arquivo(
  project_folder,
  "teste.txt",
  "Olá mundo, estou aprendendo a usar o R nas minhas aulas de Residência, isso não é demais?!"
)

Local file:

• /tmp/RtmpF2TyOr/file74d3521b984

Uploaded into Drive file:

• teste.txt <id: 19YKQLc7EGL-T5LDJtFXJ1RIuT8sQYD9L>

With MIME type:

• text/plain



In [13]:
#CRUD
#R

ler_arquivo <- function(pasta, nome) {
  arquivos <- drive_ls(pasta)
  arq <- arquivos[arquivos$name == nome, ]

  if (nrow(arq) == 0) stop("Arquivo não encontrado")

  temp <- tempfile()
  drive_download(arq$id, path = temp, overwrite = TRUE)

  readLines(temp)
}

ler_arquivo(project_folder, "teste.txt")

File downloaded:

• teste.txt <id: 19YKQLc7EGL-T5LDJtFXJ1RIuT8sQYD9L>

Saved locally as:

• /tmp/RtmpF2TyOr/file74d8e12ca1



[1] "Olá mundo, estou aprendendo a usar o R nas minhas aulas de Residência, isso não é demais?!"

In [14]:
#CRUD
#U

atualizar_arquivo <- function(pasta, nome, conteudo) {
  arquivos <- drive_ls(pasta)
  arq <- arquivos[arquivos$name == nome, ]

  if (nrow(arq) == 0) stop("Arquivo não encontrado")

  temp <- tempfile()
  writeLines(conteudo, temp)

  drive_update(arq$id, media = temp)

  unlink(temp)
}

atualizar_arquivo(
  project_folder,
  "teste.txt",
  "Agora eu atualizei o conteúdo do arquivo 😄"
)

File updated:

• teste.txt <id: 19YKQLc7EGL-T5LDJtFXJ1RIuT8sQYD9L>



In [15]:
#CRUD
#D

deletar_arquivo <- function(pasta, nome) {
  arquivos <- drive_ls(pasta)
  arq <- arquivos[arquivos$name == nome, ]

  if (nrow(arq) == 0) {
    stop("Arquivo não encontrado")
  }

  drive_trash(arq$id)

  message(paste("Arquivo", nome, "deletado"))
}

deletar_arquivo(project_folder, "teste.txt")

File trashed:

• teste.txt <id: 19YKQLc7EGL-T5LDJtFXJ1RIuT8sQYD9L>

Arquivo teste.txt deletado



In [16]:
ler_arquivo(project_folder, "teste.txt")

ERROR: Error in ler_arquivo(project_folder, "teste.txt"): Arquivo não encontrado


# Explicação do SCRIPT

O script original (Criado na sala para tentativa de acesso ao drive com auxilio da deepseeak) foi refatorado com o objetivo de torná-lo modular e executável por partes no Colab. Para isso, foram separadas as etapas: instalação de pacotes, autenticação no Google Drive, configuração da pasta de trabalho e as operações CRUD (Create, Read, Update e Delete).